# Model Informations

In [ ]:
# params
k = 2, l = 2

# iteration 1

dataset = [
  ['A', 'B', 'C', 'y'],
  [ 0 ,  1 ,  0 ,  0 ],
  [ 1 ,  0 ,  1 ,  1 ],
  [ 0 ,  1 ,  0 ,  1 ],
  [ 0,   0,   0,   1 ]
]

rules = ('A' and '¬B')

''' propositional variables to create the rule
bA,1,1 = 1 bA,1,2 = 0
bB,1,1 = 0 bB,1,2 = 1
bC,1,1 = 0 bC,1,2 = 0

u*1,1 = 0  u*1,2 = 0
p1,1 = 1   p1,2 = 0
'''

# iteration 2

dataset = [
  ['A', 'B', 'C', 'y'],
  [ 0 ,  1 ,  0 ,  1 ],
  [ 0,   0,   0,   1 ]
]

# OBS:
''' restrictions to create an inconsistent rule starting from the second iteration

- for each rule that already exists in the set of rules do:
(bA,1,1 and ¬p1,1) or 
(bA,1,2 and ¬p1,2) or 
(bB,1,1 and p1,1) or 
(bB,1,2 and p1,2)

- converting to CNF using Tseytin
(x or y or z or w) and

//** x -> (bA,1,1 and ¬p1,1) **//
(¬x or bA,1,1) and
(¬x or ¬p1,1) and

(¬y or bA,1,2) and
(¬y or ¬p1,2) and

(¬z or bB,1,1) and
(¬z or p1,1) and

(¬w or bB,1,2) and
(¬w or p1,2)

'''

# Model Tests

In [1]:
import sys, os
if not sys.path[0] == os.path.abspath('..'):
    sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from models.imlib import IMLIB
from models.i_imlib import I_IMLIB
from models.di_imlib import DI_IMLIB
from models.di_imlib_m import DI_IMLIB_M
from sklearn.model_selection import train_test_split


Xy = pd.read_csv('../databases/iris.csv')

X = Xy.drop(['Class'], axis=1)
y = Xy['Class']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2)

In [5]:
di_model = DI_IMLIB(
    max_rule_set_size=3,
    max_size_each_rule=4,
    categorical_columns_index=[],
    number_quantiles_ordinal_columns=5,
    number_lines_per_partition=8,
    rules_accuracy_weight=10,
    balance_instances=True,
    # balance_instances_seed=21
)
# di_m_model = DI_IMLIB_M(
#     max_rule_set_size=3,
#     max_size_each_rule=4,
#     categorical_columns_index=[],
#     number_quantiles_ordinal_columns=4,
#     number_lines_per_partition=8,
#     rules_accuracy_weight=10,
#     balance_instances=True,
#     # balance_instances_seed=21
# )

di_model.fit(X, y)
# di_m_model.fit(X,y)

In [14]:
line_instance = 55
instance = Xy.iloc[line_instance - 2].values[:-1]
print(f'Rules: {di_model.get_rules()}')
print(f'Instance: {[feat + ": " + str(instance[i_feat]) for i_feat, feat in enumerate(Xy.columns.values[:-1])]}')
print(f'Predict: {di_model.predict(instance)}')
print(f'Sufficient reason: {di_model.get_sufficient_reasons(instance)}')

Rules: (petal-length <= 5.32 and petal-length > 1.5 and petal-width <= 1.5 and petal-width > 1.1600000000000001)
Instance: ['sepal-length: 5.5', 'sepal-width: 2.3', 'petal-length: 4.0', 'petal-width: 1.3']
Predict: 1
Sufficient reason: (petal-width > 1.1600000000000001 and petal-length > 1.5 and petal-length <= 5.32 and petal-width <= 1.5)


In [ ]:
'(petal-length > 3.9) or '
'(petal-length <= 3.9)'

In [10]:
di_model._DI_IMLIB__rules_columns

[[-6, 15, -13]]

In [10]:
di_model.get_dataset_binarized().get_original_to_binarized_values()[0]['John']

array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 0])

In [22]:
di_model.get_dataset_binarized().get_qtts_binarized_feat_per_original_feat()

<IntegerArray>
[4, 4, 4, 4]
Length: 4, dtype: Int64

In [23]:
di_model.get_dataset_binarized().get_binarized_columns_positions()

<NumpyExtensionArray>
[[1, 2, 3, 4], [5, 6, 7, 8], [9, 10, 11, 12], [13, 14, 15, 16]]
Length: 4, dtype: object

In [9]:
from utils.functions import generate_consistent_assignments


assignments = generate_consistent_assignments(
    vars_list=[1, 2],
    binarized_columns_positions=[[1, 2]],
    categorical_columns_index=[]
)

for assignments in assignments:
    print(assignments)

print()

assignments = generate_consistent_assignments(
    vars_list=[2],
    binarized_columns_positions=[[1, 2], [3]],
    categorical_columns_index=[0]
)

for assignments in assignments:
    print(assignments)

{1: 0, 2: 0}
{1: 0, 2: 1}
{1: 1, 2: 1}

{2: 0}
{2: 1}


In [16]:
display(di_model.get_dataset_binarized().get_opposite_features_label()[110:])

<NumpyExtensionArray>
[    'Not Name Alec ',      'Not Name Alex',      'Not Name Anna',
    'Not Name Barbra',   'Not Name Barbra ',    'Not Name Camela',
 'Not Name Charlize ', 'Not Name Charlton ', 'Not Name Cristiano',
    'Not Name Diane ',
 ...
      'Smokes > 20.0',      'Smokes > 20.0',        'AreaQ > 3.0',
        'AreaQ > 5.0',        'AreaQ > 6.0',        'AreaQ > 8.0',
       'Alkhol > 1.0',       'Alkhol > 2.0',       'Alkhol > 4.0',
       'Alkhol > 5.0']
Length: 114, dtype: object

In [39]:
print('DI-IMLIB:')
print(di_model.get_rules())
print()
# print('DI-IMLIB-M:')
# print(di_m_model.get_rules())

DI-IMLIB:
(Not Name Alex and Alkhol > 3.0)



In [8]:
print('DI-IMLIB:', di_model.score(X, y))
# print('DI-IMLIB-M:', di_m_model.score(X, y))

DI-IMLIB: 0.4915254237288136


In [5]:
print(di_model.get_rules_size())
print(di_m_model.get_rules_size())

[4]
{np.int64(0): [1, 2], np.int64(1): [4]}


In [17]:
print(di_model.get_rule_set_size())
print(di_m_model.get_rule_set_size())

2
{np.int64(0): 2, np.int64(1): 1}


In [18]:
print(di_model.get_larger_rule_size())
print(di_m_model.get_larger_rule_size())

3
{np.int64(0): 2, np.int64(1): 3}


In [19]:
print(di_model.get_sum_rules_size())
print(di_m_model.get_sum_rules_size())

5
{np.int64(0): 3, np.int64(1): 3}


In [33]:
import re

def remove_redundancias(literals):
    parsed = []
    for literal in literals:
        match = re.match(r"(\w+)\s*([<>]=?)\s*(-?\d+)", literal)
        if match:
            var, op, value = match.groups()
            value = int(value)
            parsed.append((var, op, value, literal))

    reduced = {}
    
    for var, op, value, literal in parsed:
        if var not in reduced:
            reduced[var] = []
        reduced[var].append((op, value, literal))

    final_literals = set(literals)
    
    for var, conditions in reduced.items():
        conditions.sort(key=lambda x: x[1])  # Ordena pelo valor numérico
        
        to_remove = set()
        for i in range(len(conditions) - 1):
            op1, val1, lit1 = conditions[i]
            op2, val2, lit2 = conditions[i + 1]

            if op1 == "<=" and op2 == "<=":
                to_remove.add(lit2)
            elif op1 == ">=" and op2 == ">=":
                to_remove.add(lit1)
            elif op1 == ">" and op2 == ">":
                to_remove.add(lit1)
            elif op1 == "<" and op2 == "<":
                to_remove.add(lit2)

        final_literals -= to_remove
    
    return list(final_literals)

# Exemplo de uso:
literals = ['idade <= 20', 'idade <= 16.9', 'idade <= 17', 'salario > 300', 'salario > 500']
print(remove_redundancias(literals))

['idade <= 16.9', 'salario > 500']
